# AMPS I · Week 06 — HW6: Multi-material problems and Robin conditions
**Due:** before the Week 7 lecture · **Points:** 100 · **Submit:** this notebook (`.ipynb`) with all outputs

**Rules**
- Before submitting: *Runtime → Restart session and run all*. Notebooks that do not run end to end receive 0.
- An error inside a `%%fenicsx` cell does **not** stop *Run all*. Read every output before you submit.
- LLM assistants are encouraged. Fill in the **AI-usage log** at the end.
- Every number you report must come from a cell you ran.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## Q1 — Proofs (25 pts)
$\Omega\subset\mathbb R^2$ a convex polygon, $V=H^1(\Omega)$ with $\|v\|^2=\|v\|^2_{L^2}+|v|^2_{H^1}$. You may use the mean estimate
$\|v-\bar v\|_{L^2}\le C_M|v|_{H^1}$ and the trace inequality $\|v\|_{L^2(\partial\Omega)}\le C_T\|v\|$ (lecture §A4, §A6).

**(a) (8) Pure Neumann with boundary data.** $b(u,v)=\int\nabla u\cdot\nabla v$, $F(v)=\int_\Omega fv+\int_{\partial\Omega}gv$ with $f\in L^2(\Omega)$,
$g\in L^2(\partial\Omega)$. (i) Show that $F$ is bounded on $V$ and give a bound for $\|F\|_{V'}$. (ii) Show that $\int f+\int g=0$ is necessary for a
solution to exist. (iii) Show that on $\bar V=\{v:\int v=0\}$ the problem has a unique solution, and that it satisfies $b(u,v)=F(v)$ for all
$v\in V$ if the data are compatible. Give the stability bound.

**(b) (9) Robin.** $b(u,v)=\int\nabla u\cdot\nabla v+\alpha\int_{\partial\Omega}uv$ with a constant $\alpha>0$. (i) Prove the Friedrichs-type inequality
$\|v\|^2_{L^2}\le C_F\bigl(|v|^2_{H^1}+\|v\|^2_{L^2(\partial\Omega)}\bigr)$ *(hint: split $v=(v-\bar v)+\bar v$ and write $\bar v$ through a boundary integral)*.
(ii) Derive explicit continuity and coercivity constants $M(\alpha)$ and $\gamma(\alpha)$ in terms of $C_T$, $C_F$, $\alpha$. (iii) How does your $\gamma(\alpha)$
behave as $\alpha\to0$? Compare with the numbers you compute in Q3(a).

**(c) (8) Lifting in matrix form.** Order the DOFs as free ($I$) and Dirichlet ($D$) and let $\mathbf u_D=\mathbf g$. (i) Show that the Galerkin
equations are $A_{II}\mathbf u_I=\mathbf b_I-A_{ID}\mathbf g$, and that DOLFINx's system
$\begin{bmatrix}A_{II}&0\\0&I\end{bmatrix}\mathbf u=\begin{bmatrix}\mathbf b_I-A_{ID}\mathbf g\\\mathbf g\end{bmatrix}$ has the same solution and is symmetric.
(ii) Another common recipe zeroes only the **rows** of $D$ (diagonal 1, $\mathbf b_D=\mathbf g$) and keeps the columns. Show that it gives the
same solution without any lifting but a non-symmetric matrix. (iii) Explain why DOLFINx *without* `apply_lifting` gives a wrong solution.

> *your proofs*

## Q2 — A composite wall with a convective boundary (35 pts)
Steady heat conduction in a wall $\Omega=(0,1)^2$ made of two layers, $k=k_1=0.5$ for $x<s$ and $k=k_2=5$ for $x>s$:
$$-\nabla\cdot(k\nabla T)=Q\,\chi_{\{x<s\}},\qquad T=T_0=100\ \text{on }x=0,\qquad -k\,\partial_nT=h\,(T-T_\infty)\ \text{on }x=1,\qquad \partial_nT=0\ \text{on }y=0,1,$$
with $h=2$, $T_\infty=20$, and heat generation $Q$ in layer 1 only.

**(a) (12)** Derive the weak form (which terms go into $a$, which into $L$?) and the exact solution (it depends on $x$ only; flux $q=-kT'$ is
continuous). Implement with facet tags, `ds(i)`, a DG0 conductivity and `-np 2`. For $Q=0$, $s=\tfrac12$, P1, $N=8,16,32,64$: report the $L^2$ error
and the heat flux $\int_{x=1}h(T_h-T_\infty)\,ds$ against the exact value. Why is the error at round-off level?

**(b) (10)** $Q=200$, $s=\tfrac12$: P1 and P2, $N=8,\dots,64$. Tabulate $L^2$/$H^1$ errors and rates and the flux. Explain the P2 result, and plot
$T_h$ and $T$ along $y=\tfrac12$ (P1, $N=8$).

**(c) (8)** Now $s=\tfrac13$ with $N=8,16,32,64$ (the interface cuts cells) and with $N=9,18,36,72$ (it lies on mesh lines), P1 and P2. Rates? Flux error?
What should you do in practice?

**(d) (5)** $Q=200$, $s=\tfrac12$, P2, $N=16$: $T_h(1,\tfrac12)-T_\infty$ for $h=1,10,10^2,10^3,10^4$. How fast does the Robin solution approach the
Dirichlet solution $T(1)=T_\infty$? Explain with the exact solution.

In [ ]:
%%fenicsx -np 2
# TODO: (a) weak form, exact solution, solver; Q = 0, s = 1/2, P1, N = 8..64: L2 error and flux

> *your weak form, exact solution and explanation*

In [ ]:
%%fenicsx -np 2
# TODO: (b) Q = 200, s = 1/2: P1 and P2 errors, rates and flux; save the P1 N = 8 profile along y = 1/2 to /tmp/fnx/hw6_profile.npz

In [ ]:
# TODO: plot T_h (dots) and the exact T (line) along y = 1/2 from /tmp/fnx/hw6_profile.npz

> *your explanation of (b)*

In [ ]:
%%fenicsx -np 2
# TODO: (c) s = 1/3 with N = 8..64 (cut) and N = 9..72 (fitted), P1 and P2: errors, rates, flux error

> *your explanation of (c)*

In [ ]:
%%fenicsx -np 2
# TODO: (d) T_h(1, 1/2) - T_inf for h = 1 ... 1e4 and the exact value

> *your explanation of (d)*

## Q3 — Coercivity constants for Robin and for one pinned DOF (20 pts)
P1 on the unit square, dense matrices on one process.

**(a) (8)** $\gamma_h=\lambda_{\min}(K+\alpha B,G)$ for the Robin form, $\alpha=10^{-2},10^{-1},1,10,10^2,10^4$, $N=8,16,32$, and the trace constant
$C_{T,h}^2=\lambda_{\max}(B,G)$. Explain both limits $\alpha\to0$ and $\alpha\to\infty$ (give the numbers).

**(b) (12)** Poisson with a single DOF clamped to zero: remove one row and column from $K$ and $G$ and compute $\gamma_h$ for the DOF at the corner
$(0,0)$, the edge midpoint $(\tfrac12,0)$ and the centre $(\tfrac12,\tfrac12)$, $N=4,8,\dots,64$ (use `scipy.linalg.eigh(..., subset_by_index=[0, 0])`).
Tabulate $1/\gamma_h$ and its increments. Show that the increments approach $\log 2/\theta$ with $\theta$ the angle of $\Omega$ at the point.
What does this mean for "pinning one DOF" as a boundary condition, and how is it related to HW5 Q3(b)?

In [ ]:
%%fenicsx -np 1
# TODO: (a) Robin gamma_h and the trace constant

> *your explanation of (a)*

In [ ]:
%%fenicsx -np 1
# TODO: (b) one pinned DOF at the corner, an edge midpoint and the centre

> *your explanation of (b)*

## Q4 — LLM-generated mixed-BC solver with planted bugs (20 pts)
The code below was produced by an LLM for the prompt *"DOLFINx 0.11 MMS study for $-\Delta u=f$ on the unit square with $u=\tilde u$ on $x=0$,
$\partial_nu=g$ on $y=0,1$ and $\partial_nu+10u=r$ on $x=1$; P1, $N=8,16,32,64$, `mpirun -np 2`"*. It contains **at least six** mistakes (API,
boundary location, measures, data, Robin form, lifting and parallel ordering).

**(a) (12)** Fix it in the next code cell, so that it runs and reports rates 2 and 1.
**(b) (8)** Table: for each mistake give the line, what was wrong, how you detected it (error message, rate, a printed check, `-np 1` vs `-np 2`, ...),
and the fix.

*The first cell is meant to fail. The real error is the last traceback (printed once per rank).*

In [ ]:
%%fenicsx -np 2
# EXPECTED-FAIL — the code as generated (do not edit this cell)
from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import assemble_matrix, assemble_vector, apply_lifting, set_bc

def solve_mms(N, alpha=10.0):
    msh = mesh.create_unit_square(MPI.COMM_WORLD, N, N)
    fdim = msh.topology.dim - 1
    right = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], 1))
    bottom = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[1], 0))
    top = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[1], 1))
    facets = np.hstack([right, bottom, top]).astype(np.int32)
    tags = np.hstack([np.full(len(right), 2), np.full(len(bottom), 3), np.full(len(top), 4)]).astype(np.int32)
    order = np.argsort(facets)
    ft = mesh.meshtags(msh, fdim, facets[order], tags[order])
    ds = ufl.Measure("ds", domain=msh)
    V = fem.functionspace(msh, ("Lagrange", 1))
    x = ufl.SpatialCoordinate(msh)
    u_ex = 1 + ufl.sin(ufl.pi * x[0]) * ufl.exp(x[1]) + x[0] * x[1] ** 2
    f = -ufl.div(ufl.grad(u_ex))
    g = u_ex.dx(1)                                   # du/dn on the top and bottom sides
    r = u_ex.dx(0) + alpha * u_ex                    # du/dn + alpha u on the right side
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    a = fem.form(ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx)
    L = fem.form(f * v * ufl.dx + g * v * ds(3) + g * v * ds(4) + r * v * ds(2))
    uD = fem.Function(V)
    uD.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    dofs = fem.locate_dofs_geometrical(V, lambda x: x[0] == 0.0)
    bc = fem.dirichletbc(uD, dofs, V)
    A = assemble_matrix(a, bcs=[bc]); A.assemble()
    b = assemble_vector(L)
    set_bc(b, [bc])
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    ksp = PETSc.KSP().create(msh.comm)
    ksp.setOperators(A); ksp.setType("preonly"); ksp.getPC().setType("lu")
    uh = fem.Function(V)
    ksp.solve(b, uh.x.petsc_vec); uh.x.scatter_forward()
    e = uh - u_ex
    L2 = np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(e * e * ufl.dx)), op=MPI.SUM))
    H1 = np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(ufl.inner(ufl.grad(e), ufl.grad(e)) * ufl.dx)), op=MPI.SUM))
    return L2, H1

Ns = np.array([8, 16, 32, 64])
E = np.array([solve_mms(N) for N in Ns])
rates = np.log2(E[:-1] / E[1:])
if MPI.COMM_WORLD.rank == 0:
    for N, (l2, h1), r in zip(Ns, E, [(np.nan, np.nan)] + list(rates)):
        print(f"N={N:3d}  L2 {l2:.3e}  H1 {h1:.3e}   rates {r[0]:.2f} {r[1]:.2f}")

In [ ]:
%%fenicsx -np 2
# TODO: your corrected version

| # | line | mistake | how detected | fix |
|---|---|---|---|---|
| 1 | | | | |

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …